# Proyecto final de aprendizaje automático

In [36]:
import pandas as pd
import sqlite3

## Paso 1: Definición del problema

### Objetivo: 
- Predecir la demanda futura de productos farmacéuticos para optimizar la gestión de inventario.
- Evitar quiebres de stock y sobrestock. 
- Reducir pérdida por vencimientos y mejorar disponibilidad.
##### Plan B: Predecir el total que abonará el cliente en una transacción, considerando cantidad, producto, rubro, cobertura, descuentos e impuestos.

## Paso 2: Obtencion y carga del conjunto de datos

In [37]:
df = pd.read_csv('../data/raw/farmacia-datos.csv', sep=';',encoding='latin-1')
df.head(12)

,Fecha,Tipo Mov.,Fac. Tipo,Fac. Suc.,Fac. Nun.,Fisc. Numero,Tipo Pago,Cant.,Precio,Producto,Sub. Total,Rubro,Cobertura,Ajustes,Desc. Adic.,Total. Cliente,IVA,Tasa Iva,Total Gravado,Total sin Gravar
0,01/01/25 01.13.46,F,B,0,379923,NaN,E,1,9344,ACTRON PEDIATRICO 4% susp.oral x 100 ml,9344,FARMACIA,0,0,0,9344,0,0,0,9344
1,01/01/25 01.59.21,F,B,0,379924,NaN,E,1,"8174,96",AMOXIDAL 500 mg comp.rec.x 21,"8174,96",FARMACIA,"3032,85",0,0,"5142,11",0,0,0,"8174,96"
2,01/01/25 02.01.59,F,B,0,379925,NaN,E,1,"8174,96",AMOXIDAL 500 mg comp.rec.x 21,"8174,96",FARMACIA,"5318,1",0,0,"2856,86",0,0,0,"8174,96"
3,01/01/25 02.04.14,F,B,0,379926,NaN,E,1,"8086,82",MUELITA FORTE GEL gel pomo x 10 g,"8086,82",FARMACIA,0,0,0,"8086,82",0,0,0,"8086,82"
4,01/01/25 02.08.35,F,B,0,379927,NaN,E,1,"26750,12",DIOXAFLEX B12 comp.x 20,"26750,12",FARMACIA,0,0,0,"26750,12",0,0,0,"26750,12"
5,01/01/25 06.06.42,F,B,0,379928,NaN,E,1,"24480,51",EVATEST DIGITAL env.x 1,"24480,51",FARMACIA,0,0,0,"24480,51","4248,683554",21,"20231,82645",0
6,01/01/25 06.22.37,F,B,0,379929,NaN,E,1,4800,GEN LP KETOROLAC blister 20 mg x 10,4800,FARMACIA,0,0,0,4800,0,0,0,4800
7,01/01/25 06.22.37,F,B,0,379929,NaN,E,1,4961,CAFIASPIRINA PLUS comp.x 20,4961,FARMACIA,0,0,0,4961,0,0,0,4961
8,01/01/25 07.46.36,F,B,0,379930,NaN,E,2,3500,ACCESORIO (unica),7000,FARMACIA,0,0,0,7000,"1214,876033",21,"5785,123967",0
9,02/01/25 08.08.38,F,B,0,379931,NaN,E,1,3430,QURA PLUS comp.rec.x 20,3430,FARMACIA,0,0,343,3087,0,0,0,3430


> Los datos provienen de un archivo Excel extraído del sistema de facturación de una farmacia real, convertido posteriormente a formato CSV para su procesamiento.

## Paso 3: Almacenar la información

In [38]:
df.columns = df.columns.str.replace('.', '', regex=False).str.replace(' ', '_')

Para la ejecución de las consultas SQL se normalizaron los nombres de columnas para garantizar la correcta agregación de variables numéricas.

In [39]:
# Create connection to SQLite
conn = sqlite3.connect('farmacia-datos.db')

In [40]:
# Save table
df.to_sql('ventas', conn, if_exists='replace', index=False)

117415

In [41]:
# Verify that the table exists
query = 'SELECT * FROM ventas LIMIT 5;'
pd.read_sql(query, conn)

,Fecha,Tipo_Mov,Fac_Tipo,Fac_Suc,Fac_Nun,Fisc_Numero,Tipo_Pago,Cant,Precio,Producto,Sub_Total,Rubro,Cobertura,Ajustes,Desc_Adic,Total_Cliente,IVA,Tasa_Iva,Total_Gravado,Total_sin_Gravar
0,01/01/25 01.13.46,F,B,0,379923,None,E,1,9344,ACTRON PEDIATRICO 4% susp.oral x 100 ml,9344,FARMACIA,0,0,0,9344,0,0,0,9344
1,01/01/25 01.59.21,F,B,0,379924,None,E,1,"8174,96",AMOXIDAL 500 mg comp.rec.x 21,"8174,96",FARMACIA,"3032,85",0,0,"5142,11",0,0,0,"8174,96"
2,01/01/25 02.01.59,F,B,0,379925,None,E,1,"8174,96",AMOXIDAL 500 mg comp.rec.x 21,"8174,96",FARMACIA,"5318,1",0,0,"2856,86",0,0,0,"8174,96"
3,01/01/25 02.04.14,F,B,0,379926,None,E,1,"8086,82",MUELITA FORTE GEL gel pomo x 10 g,"8086,82",FARMACIA,0,0,0,"8086,82",0,0,0,"8086,82"
4,01/01/25 02.08.35,F,B,0,379927,None,E,1,"26750,12",DIOXAFLEX B12 comp.x 20,"26750,12",FARMACIA,0,0,0,"26750,12",0,0,0,"26750,12"


**Top 10 productos más vendidos**

In [42]:
# Execute a real SQL query
query_top_10 = '''
SELECT Producto, SUM("Cant") AS Demanda_total
FROM ventas
GROUP BY Producto
ORDER BY Demanda_total DESC
LIMIT 10;
'''
pd.read_sql(query_top_10, conn)

,Producto,Demanda_total
0,BUSCAPINA COMPOSITUM comp.rec.x 20,2648
1,QURA PLUS comp.rec.x 20,2602
2,GEN LP IBUPROFENO blister 600 mg x 10,2135
3,SERTAL COMPUESTO comp.rec.x 20,1786
4,GEN LP OMEPRAZOL blister 20 mg x 15,1576
5,ALIKAL (unica),1483
6,MYLANTA EXTRA comp.mast.x 24,1278
7,ASPIRINA PREVENT comp.cub.enterica x 50,1160
8,CAFIASPIRINA comp.x 30,1041
9,TAFIROL 1G X 8 (unica),983


> Se almacenaron los datos en una base de datos SQLite y se realizaron consultas SQL desde Python para identificar productos de alta rotación y patrones de demanda.

**Demanda total por producto**

In [43]:
query_product = '''
SELECT Producto, SUM("Cant") AS Demanda_total
FROM ventas
GROUP BY Producto;
'''
pd.read_sql(query_product, conn)

,Producto,Demanda_total
0,None,1
1,CHUP.MANZANITA(C/DIBUJO)T/SILIC.+3M RED - ES...,1
2,(unica),4
3,1,1
4,1 1,7
...,...,...
7671,q,1
7672,|,1
7673,º,3
7674,ÓLEO Calcáreo x 240 ml.,1


Esta consulta permite identificar los productos con mayor volumen de ventas acumuladas, fundamentales para la gestión de stock y la priorización de reposición.

**Demanda agregada por día**

In [44]:
query_per_day = """
SELECT strftime('%Y-%m', Fecha) AS mes, SUM("Cant") AS Demanda_mensual
FROM ventas
GROUP BY mes
ORDER BY mes;
"""

pd.read_sql(query_per_day, conn)

,mes,Demanda_mensual
0,None,160639


La agregación diaria por producto permite modelar la demanda como una serie temporal, base para la predicción de consumo futuro.

In [45]:
query_coverage = """
SELECT Cobertura, SUM("Cant") AS Demanda_total
FROM ventas
GROUP BY Cobertura;
"""
pd.read_sql(query_coverage, conn)


,Cobertura,Demanda_total
0,"-157953,92",1
1,0,107170
2,10000,1
3,"10000,188",1
4,"10001,04",3
...,...,...
23016,"9993,308",1
23017,"9995,3",3
23018,"9998,392",2
23019,"9998,952",1


La presencia de cobertura médica incrementa la demanda, lo cual debe considerarse en la planificación de inventario.

## Paso 4: Realiza un análisis descriptivo

Variable clave: `Cant`

In [46]:
df['Cant'].describe()

count    117415.000000
mean          1.368130
std           1.862115
min           0.000000
25%           1.000000
50%           1.000000
75%           1.000000
max         106.000000
Name: Cant, dtype: float64

In [47]:
media = df['Cant'].mean()
mediana = df['Cant'].median()
moda = df['Cant'].mode()[0]
varianza = df['Cant'].var()
asimetria = df['Cant'].skew()

In [48]:
print(f'Media:     {media:.2f}')
print(f'Mediana:   {mediana:.2f}')
print(f'Moda:      {moda:.2f}')
print(f'Varianza:  {varianza:.2f}')
print(f'Asimetría: {asimetria:.2f}')

Media:     1.37
Mediana:   1.00
Moda:      1.00
Varianza:  3.47
Asimetría: 12.14


- **Media** (1.37) vs **Mediana** (1.00): El hecho de que la media sea mayor que la mediana indica que la distribución está sesgada a la derecha. Mientras que la mayoría de los clientes compran solo 1 unidad (mediana), hay un grupo pequeño de transacciones con cantidades muy altas que elevan el promedio.
- **Moda** (1.00): Es el valor más frecuente. Esto confirma que el comportamiento estándar en la farmacia es la compra de una única unidad por producto (típico de medicamentos bajo receta).
- **Varianza** (3.47): La desviación estándar sería la raíz cuadrada de esto (≈1.86). Aunque la mayoría compra 1 unidad, hay una variabilidad considerable. En términos de negocio, esto sugiere que conviven dos tipos de clientes: el consumidor final (compra 1 unidad) y posiblemente clientes institucionales o crónicos (compran varias cajas o tratamientos completos).
- **Asimetría Positiva Extrema**: Un valor de 12.14 es extremadamente alto (en una distribución normal sería 0).   

Debido a esta asimetría, será necesario decidir si se eliminan esos valores extremos para entrenar el modelo de Machine Learning o si se tratan de forma especial, ya que podrían distorsionar las predicciones.

In [49]:
products_no_name = df[df['Producto'].str.len() <= 4]['Producto'].unique()
products_no_name

array(['1 1', 'A1 1', '5200', 'º', '|', 'LIMA', 'q', '1'], dtype=object)

In [50]:
df.shape

(117415, 20)

In [51]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 117415 entries, 0 to 117414
Data columns (total 20 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   Fecha             117415 non-null  object 
 1   Tipo_Mov          117415 non-null  object 
 2   Fac_Tipo          117415 non-null  object 
 3   Fac_Suc           117415 non-null  int64  
 4   Fac_Nun           117415 non-null  int64  
 5   Fisc_Numero       0 non-null       float64
 6   Tipo_Pago         117415 non-null  object 
 7   Cant              117415 non-null  int64  
 8   Precio            117415 non-null  object 
 9   Producto          117414 non-null  object 
 10  Sub_Total         117415 non-null  object 
 11  Rubro             117415 non-null  object 
 12  Cobertura         117415 non-null  object 
 13  Ajustes           117415 non-null  int64  
 14  Desc_Adic         117415 non-null  object 
 15  Total_Cliente     117415 non-null  object 
 16  IVA               11

In [58]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Fac_Suc,117415.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0
Fac_Nun,117415.0,417643.591407,22420.160070,379923.0,398031.0,416281.0,437618.5,455865.0
Fisc_Numero,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Cant,117415.0,1.368130,1.862115,0.0,1.0,1.0,1.0,106.0
Ajustes,117415.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0
Tasa_Iva,117415.0,4.090363,8.316679,0.0,0.0,0.0,0.0,21.0


In [59]:
df['Fecha'] = pd.to_datetime(df['Fecha'], format='%d/%m/%y %H.%M.%S')

In [60]:
df.head(5)

,Fecha,Tipo_Mov,Fac_Tipo,Fac_Suc,Fac_Nun,Fisc_Numero,Tipo_Pago,Cant,Precio,Producto,Sub_Total,Rubro,Cobertura,Ajustes,Desc_Adic,Total_Cliente,IVA,Tasa_Iva,Total_Gravado,Total_sin_Gravar
0,2025-01-01 01:13:46,F,B,0,379923,NaN,E,1,9344,ACTRON PEDIATRICO 4% susp.oral x 100 ml,9344,FARMACIA,0,0,0,9344,0,0,0,9344
1,2025-01-01 01:59:21,F,B,0,379924,NaN,E,1,"8174,96",AMOXIDAL 500 mg comp.rec.x 21,"8174,96",FARMACIA,"3032,85",0,0,"5142,11",0,0,0,"8174,96"
2,2025-01-01 02:01:59,F,B,0,379925,NaN,E,1,"8174,96",AMOXIDAL 500 mg comp.rec.x 21,"8174,96",FARMACIA,"5318,1",0,0,"2856,86",0,0,0,"8174,96"
3,2025-01-01 02:04:14,F,B,0,379926,NaN,E,1,"8086,82",MUELITA FORTE GEL gel pomo x 10 g,"8086,82",FARMACIA,0,0,0,"8086,82",0,0,0,"8086,82"
4,2025-01-01 02:08:35,F,B,0,379927,NaN,E,1,"26750,12",DIOXAFLEX B12 comp.x 20,"26750,12",FARMACIA,0,0,0,"26750,12",0,0,0,"26750,12"


In [61]:
df.duplicated().sum()

np.int64(121)

In [62]:
duplicated = df[df.duplicated(keep=False)]
duplicated

,Fecha,Tipo_Mov,Fac_Tipo,Fac_Suc,Fac_Nun,Fisc_Numero,Tipo_Pago,Cant,Precio,Producto,Sub_Total,Rubro,Cobertura,Ajustes,Desc_Adic,Total_Cliente,IVA,Tasa_Iva,Total_Gravado,Total_sin_Gravar
2122,2025-01-07 14:56:11,F,B,0,381215,NaN,E,1,3200,IMPULSE VERY PINK BS AERO 150,3200,PERFUMERIA,0,0,0,3200,"555,3719008",21,"2644,628099",0
2123,2025-01-07 14:56:11,F,B,0,381215,NaN,E,1,3200,IMPULSE VERY PINK BS AERO 150,3200,PERFUMERIA,0,0,0,3200,"555,3719008",21,"2644,628099",0
2124,2025-01-07 14:56:11,F,B,0,381215,NaN,E,1,3200,IMPULSE VERY PINK BS AERO 150,3200,PERFUMERIA,0,0,0,3200,"555,3719008",21,"2644,628099",0
4858,2025-01-14 16:00:35,F,B,0,382867,NaN,E,1,"21971,64",MEMANTINA RICHET 10 mg comp.x 30,"21971,64",FARMACIA,"10985,82",0,0,"10985,82",0,0,0,"21971,64"
4859,2025-01-14 16:00:35,F,B,0,382867,NaN,E,1,"21971,64",MEMANTINA RICHET 10 mg comp.x 30,"21971,64",FARMACIA,"10985,82",0,0,"10985,82",0,0,0,"21971,64"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
112196,2025-12-06 11:10:48,F,B,0,452638,NaN,E,1,"8157,79",TAFIROL PLUS RAPIDA ACCION cáps.bl. x 16,"8157,79",FARMACIA,0,0,0,"8157,79",0,0,0,"8157,79"
114299,2025-12-13 09:11:38,F,B,0,453912,NaN,E,1,9358,LOTRIAL 10 mg comp.x 30,9358,FARMACIA,0,0,0,9358,0,0,0,9358
114300,2025-12-13 09:11:38,F,B,0,453912,NaN,E,1,9358,LOTRIAL 10 mg comp.x 30,9358,FARMACIA,0,0,0,9358,0,0,0,9358
115321,2025-12-16 16:31:39,F,B,0,454552,NaN,E,1,"9641,696",BAGOVIT CAPILAR PLASMA VEG 350,"9641,696",PERFUMERIA,0,0,0,"9641,696","1673,352198",21,"7968,343802",0


### Limpieza de datos: Eliminar información irrelevante

In [65]:
df['Fisc_Numero'].isna().sum()

np.int64(117415)

In [66]:
(df['Fac_Suc'] == 0).sum()

np.int64(117415)

In [72]:
df['Tipo_Mov'].unique()

array(['F'], dtype=object)

In [68]:
df['Fac_Tipo'].value_counts(dropna=False)

Fac_Tipo
B    117251
A       164
Name: count, dtype: int64

In [69]:
df['Tipo_Pago'].value_counts(dropna=False)

Tipo_Pago
E    117415
Name: count, dtype: int64

In [73]:
df['Ajustes'].value_counts(dropna=False)

Ajustes
0    117415
Name: count, dtype: int64

In [74]:
df['Desc_Adic'].value_counts(dropna=False)

Desc_Adic
0             115273
-3,64E-12        226
-1,82E-12         86
-7,28E-12         79
-9,09E-13         24
               ...  
1450               1
8741,37096         1
4704,488           1
231,56             1
1148               1
Name: count, Length: 1155, dtype: int64

- La variable `Fisc_Numero` no aporta información al análisis, ya que presenta un 100% de valores nulos (0 non-null). Al no contener observaciones válidas, se elimina del dataset para evitar ruido y simplificar el análisis.
- Las variables `Fac_Suc`,`Tipo_Mov`, `Tipo_Pago`, `Ajustes`  presentan un único valor en la totalidad de los registros. Al no mostrar variabilidad, no aporta información relevante para el análisis ni para modelos posteriores, por lo que se eliminan del dataset.

In [75]:
df.drop(['Fisc_Numero', 'Fac_Suc', 'Tipo_Mov', 'Tipo_Pago', 'Ajustes'], axis=1, inplace=True)
df

,Fecha,Fac_Tipo,Fac_Nun,Cant,Precio,Producto,Sub_Total,Rubro,Cobertura,Desc_Adic,Total_Cliente,IVA,Tasa_Iva,Total_Gravado,Total_sin_Gravar
0,2025-01-01 01:13:46,B,379923,1,9344,ACTRON PEDIATRICO 4% susp.oral x 100 ml,9344,FARMACIA,0,0,9344,0,0,0,9344
1,2025-01-01 01:59:21,B,379924,1,"8174,96",AMOXIDAL 500 mg comp.rec.x 21,"8174,96",FARMACIA,"3032,85",0,"5142,11",0,0,0,"8174,96"
2,2025-01-01 02:01:59,B,379925,1,"8174,96",AMOXIDAL 500 mg comp.rec.x 21,"8174,96",FARMACIA,"5318,1",0,"2856,86",0,0,0,"8174,96"
3,2025-01-01 02:04:14,B,379926,1,"8086,82",MUELITA FORTE GEL gel pomo x 10 g,"8086,82",FARMACIA,0,0,"8086,82",0,0,0,"8086,82"
4,2025-01-01 02:08:35,B,379927,1,"26750,12",DIOXAFLEX B12 comp.x 20,"26750,12",FARMACIA,0,0,"26750,12",0,0,0,"26750,12"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
117410,2025-12-23 10:23:06,B,455861,1,5000,COLGATE (unica),5000,FARMACIA,0,0,5000,0,0,0,5000
117411,2025-12-23 10:25:57,B,455862,1,"13641,2",TAFIROL 1 G comp.ran.x 50,"13641,2",FARMACIA,"5808,98",0,"7832,22",0,0,0,"13641,2"
117412,2025-12-23 10:34:31,B,455863,1,16000,CICATRICURE GEL Gel NF x 30 g,16000,FARMACIA,0,0,16000,"2776,859504",21,"13223,1405",0
117413,2025-12-23 10:38:49,B,455864,1,"36758,39",LOUTEN T colirio x 2.5 ml,"36758,39",FARMACIA,"14703,36",0,"22055,03",0,0,0,"36758,39"


### Análisis de Variables

In [80]:
numerical_variables = df.select_dtypes(include=['int64', 'float64'])
numerical_variables

,Fac_Nun,Cant,Tasa_Iva
0,379923,1,0
1,379924,1,0
2,379925,1,0
3,379926,1,0
4,379927,1,0
...,...,...,...
117410,455861,1,0
117411,455862,1,0
117412,455863,1,21
117413,455864,1,0


In [81]:
categorical_variables = df.select_dtypes(include=['object'])
categorical_variables

,Fac_Tipo,Precio,Producto,Sub_Total,Rubro,Cobertura,Desc_Adic,Total_Cliente,IVA,Total_Gravado,Total_sin_Gravar
0,B,9344,ACTRON PEDIATRICO 4% susp.oral x 100 ml,9344,FARMACIA,0,0,9344,0,0,9344
1,B,"8174,96",AMOXIDAL 500 mg comp.rec.x 21,"8174,96",FARMACIA,"3032,85",0,"5142,11",0,0,"8174,96"
2,B,"8174,96",AMOXIDAL 500 mg comp.rec.x 21,"8174,96",FARMACIA,"5318,1",0,"2856,86",0,0,"8174,96"
3,B,"8086,82",MUELITA FORTE GEL gel pomo x 10 g,"8086,82",FARMACIA,0,0,"8086,82",0,0,"8086,82"
4,B,"26750,12",DIOXAFLEX B12 comp.x 20,"26750,12",FARMACIA,0,0,"26750,12",0,0,"26750,12"
...,...,...,...,...,...,...,...,...,...,...,...
117410,B,5000,COLGATE (unica),5000,FARMACIA,0,0,5000,0,0,5000
117411,B,"13641,2",TAFIROL 1 G comp.ran.x 50,"13641,2",FARMACIA,"5808,98",0,"7832,22",0,0,"13641,2"
117412,B,16000,CICATRICURE GEL Gel NF x 30 g,16000,FARMACIA,0,0,16000,"2776,859504","13223,1405",0
117413,B,"36758,39",LOUTEN T colirio x 2.5 ml,"36758,39",FARMACIA,"14703,36",0,"22055,03",0,0,"36758,39"


In [83]:
columns_to_numeric = ['Precio', 'Sub_Total', 'Cobertura', 'Desc_Adic', 'Total_Cliente', 'IVA', 'Total_Gravado', 'Total_sin_Gravar']
df[columns_to_numeric] = df[columns_to_numeric].apply(pd.to_numeric, errors='coerce')

In [86]:
df.dtypes

Fecha               datetime64[ns]
Fac_Tipo                    object
Fac_Nun                      int64
Cant                         int64
Precio                     float64
Producto                    object
Sub_Total                  float64
Rubro                       object
Cobertura                  float64
Desc_Adic                  float64
Total_Cliente              float64
IVA                        float64
Tasa_Iva                     int64
Total_Gravado              float64
Total_sin_Gravar           float64
dtype: object

Varias variables se encontraban almacenadas como tipo object a pesar de representar valores numéricos. Se procedió a su conversión a tipo numérico y los valores no convertibles a NaN para su posterior tratamiento.